# Baseline [acc = 0.89]

Borrowed from  https://www.kaggle.com/code/cbrogan/xgboost-example-python

XGB params:
```
    max_depth=3,
    n_estimators=300,
    learning_rate=0.05
```

In [80]:
# This script shows you how to make a submission using a few
# useful Python libraries.
# It gets a public leaderboard score of 0.76077.
# Maybe you can tweak it and do better...?

import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import numpy as np

train_df_path = '/home/stas/vaML/titanic/train.csv'
test_X_path = '/home/stas/vaML/titanic/test.csv'
test_y_path = '/home/stas/vaML/titanic/gender_submission.csv'

# Load the data
train_df = pd.read_csv(train_df_path, header=0)
test_df = pd.read_csv(test_X_path, header=0)
target = pd.read_csv(test_y_path, header=0)

# We'll impute missing values using the median for numeric columns and the most
# common value for string columns.
# This is based on some nice code by 'sveitser' at http://stackoverflow.com/a/25562948
from sklearn.base import TransformerMixin
class DataFrameImputer(TransformerMixin):
    def fit(self, X, y=None):
        self.fill = pd.Series([X[c].value_counts().index[0]
            if X[c].dtype == np.dtype('O') else X[c].median() for c in X],
            index=X.columns)
        return self
    def transform(self, X, y=None):
        return X.fillna(self.fill)

feature_columns_to_use = ['Pclass','Sex','Age','Fare','Parch']
nonnumeric_columns = ['Sex']

# Join the features from train and test together before imputing missing values,
# in case their distribution is slightly different
big_X = train_df[feature_columns_to_use].append(test_df[feature_columns_to_use])
big_X_imputed = DataFrameImputer().fit_transform(big_X)

# XGBoost doesn't (yet) handle categorical features automatically, so we need to change
# them to columns of integer values.
# See http://scikit-learn.org/stable/modules/preprocessing.html#preprocessing for more
# details and options
le = LabelEncoder()
for feature in nonnumeric_columns:
    big_X_imputed[feature] = le.fit_transform(big_X_imputed[feature])

# Prepare the inputs for the model
train_X = big_X_imputed[0:train_df.shape[0]].values
test_X = big_X_imputed[train_df.shape[0]::].values
train_y = train_df['Survived']

# You can experiment with many other options here, using the same .fit() and .predict()
# methods; see http://scikit-learn.org
# This example uses the current build of XGBoost, from https://github.com/dmlc/xgboost
gbm = xgb.XGBClassifier(max_depth=3, n_estimators=300, learning_rate=0.05).fit(train_X, train_y)
predictions = gbm.predict(test_X)

# Kaggle needs the submission to have a certain format;
# see https://www.kaggle.com/c/titanic-gettingStarted/download/gendermodel.csv
# for an example of what it's supposed to look like.
submission = pd.DataFrame({ 'PassengerId': test_df['PassengerId'],
                            'Survived': predictions })


print(classification_report(predictions, target['Survived']))

              precision    recall  f1-score   support

           0       0.94      0.90      0.92       278
           1       0.82      0.89      0.85       140

    accuracy                           0.89       418
   macro avg       0.88      0.89      0.88       418
weighted avg       0.90      0.89      0.90       418



/tmp/ipykernel_3183/1571637146.py:38: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  big_X = train_df[feature_columns_to_use].append(test_df[feature_columns_to_use])


# Testing XGB - Titanic - V1 - GPT-4o pipeline [Manual fix, acc = 0.78]

## Model response

https://chatgpt.com/share/ee964de6-7607-4a98-81a2-d0757eaa3fcf

To solve the task of predicting which passengers survived the Titanic shipwreck using the XGBoost library in Python, you can follow these steps:

1. **Data Preprocessing**:
    - Handle missing values.
    - Convert categorical columns into numeric format (one-hot encoding for non-binary categorical columns, label encoding for binary columns).
    - Split the dataset into training and testing sets.

2. **Train the XGBoost model**:
    - Import the necessary libraries.
    - Define the model parameters.
    - Train the model on the training data.

3. **Evaluate the model**:
    - Make predictions on the testing data.
    - Calculate performance metrics like accuracy, precision, recall, etc.

Here is a complete Python script demonstrating these steps:

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# Assume 'titanic_data' is the DataFrame containing the Titanic dataset
# Load your dataset
# titanic_data = pd.read_csv('path_to_your_titanic_data.csv')

# For this example, let's create a mock DataFrame similar to the Titanic dataset structure
data = {
    'Name': ['Allen, Miss. Elisabeth Walton', 'Moran, Mr. James', 'Heikkinen, Miss. Laina'],
    'Sex': ['female', 'male', 'female'],
    'Ticket': ['24160', '330877', 'STON/O2. 3101282'],
    'Cabin': [np.nan, np.nan, np.nan],
    'Embarked': ['S', 'Q', 'S'],
    'PassengerId': [1, 2, 3],
    'Survived': [1, 0, 1],
    'Pclass': [1, 3, 3],
    'Age': [29, np.nan, 26],
    'SibSp': [0, 0, 0],
    'Parch': [0, 0, 0],
    'Fare': [211.3375, 8.4583, 7.9250]
}

titanic_data = pd.DataFrame(data)

# Handle missing values
# Fill missing values in 'Age' and 'Embarked' with the median and mode respectively
imputer_age = SimpleImputer(strategy='median')
titanic_data['Age'] = imputer_age.fit_transform(titanic_data[['Age']])

imputer_embarked = SimpleImputer(strategy='most_frequent')
titanic_data['Embarked'] = imputer_embarked.fit_transform(titanic_data[['Embarked']])

# Encode categorical variables
label_encoder = LabelEncoder()
titanic_data['Sex'] = label_encoder.fit_transform(titanic_data['Sex'])

one_hot_encoder = OneHotEncoder(drop='first', sparse=False)
embarked_encoded = one_hot_encoder.fit_transform(titanic_data[['Embarked']])
embarked_encoded_df = pd.DataFrame(embarked_encoded, columns=one_hot_encoder.get_feature_names_out(['Embarked']))

# Concatenate one-hot encoded columns to the main dataframe
titanic_data = pd.concat([titanic_data, embarked_encoded_df], axis=1)

# Drop the original text columns and any columns not needed for modeling
titanic_data.drop(['Name', 'Ticket', 'Cabin', 'Embarked'], axis=1, inplace=True)

# Define the feature set and the target variable
X = titanic_data.drop(['Survived', 'PassengerId'], axis=1)
y = titanic_data['Survived']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the XGBoost model
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Print classification report
print(classification_report(y_test, y_pred))

# If needed, save the model for future use
# xgb_model.save_model('xgboost_titanic_model.json')
```

### Explanation:

1. **Data Preprocessing**:
    - Missing values in the 'Age' column are filled with the median value.
    - Missing values in the 'Embarked' column are filled with the most frequent value.
    - Categorical columns ('Sex' and 'Embarked') are encoded. 'Sex' is label encoded (since it is binary), and 'Embarked' is one-hot encoded.
    - Columns not needed for modeling (like 'Name', 'Ticket', 'Cabin', and 'PassengerId') are dropped.

2. **Model Training**:
    - The dataset is split into training and testing sets.
    - An XGBoost classifier is initialized and trained on the training set.

3. **Model Evaluation**:
    - Predictions are made on the test set.
    - Model performance is evaluated using accuracy and a classification report.

This script provides a comprehensive approach to preprocessing the data, training an XGBoost model, and evaluating its performance on the Titanic survival prediction task.


## Pipeline testing

**Accuracy: 0.78**

Manual modification: dataset loading, preprocessing transformed to a function

XGB params:
```
    none
```

In [81]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# Handle missing values
def preprocess(titanic_data):
    # Fill missing values in 'Age' and 'Embarked' with the median and mode respectively
    imputer_age = SimpleImputer(strategy='median')
    titanic_data['Age'] = imputer_age.fit_transform(titanic_data[['Age']])
    
    imputer_embarked = SimpleImputer(strategy='most_frequent')
    titanic_data['Embarked'] = imputer_embarked.fit_transform(titanic_data[['Embarked']])
    
    # Encode categorical variables
    label_encoder = LabelEncoder()
    titanic_data['Sex'] = label_encoder.fit_transform(titanic_data['Sex'])
    
    one_hot_encoder = OneHotEncoder(drop='first', sparse=False)
    embarked_encoded = one_hot_encoder.fit_transform(titanic_data[['Embarked']])
    embarked_encoded_df = pd.DataFrame(embarked_encoded, columns=one_hot_encoder.get_feature_names_out(['Embarked']))
    
    # Concatenate one-hot encoded columns to the main dataframe
    titanic_data = pd.concat([titanic_data, embarked_encoded_df], axis=1)
    
    # Drop the original text columns and any columns not needed for modeling
    titanic_data.drop(['Name', 'Ticket', 'Cabin', 'Embarked'], axis=1, inplace=True)
    return titanic_data

# Assume 'titanic_data' is the DataFrame containing the Titanic dataset
# Load your dataset
train_df = pd.read_csv('/home/stas/vaML/titanic/train.csv')
test_df = pd.read_csv('/home/stas/vaML/titanic/test.csv')
res = pd.read_csv('/home/stas/vaML/titanic/gender_submission.csv')

train_df = preprocess(train_df)
test_df = preprocess(test_df)

# Define the feature set and the target variable
X_train = train_df.drop(['Survived', 'PassengerId'], axis=1)
y_train = train_df['Survived']

X_test = test_df.drop(['PassengerId'], axis=1)
y_test = res.drop(['PassengerId'], axis=1)

# Initialize and train the XGBoost model
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Print classification report
print(classification_report(y_test, y_pred))


Accuracy: 0.78
              precision    recall  f1-score   support

           0       0.86      0.79      0.82       266
           1       0.67      0.77      0.72       152

    accuracy                           0.78       418
   macro avg       0.76      0.78      0.77       418
weighted avg       0.79      0.78      0.78       418



/home/ubuntu/anaconda3/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/home/ubuntu/anaconda3/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


# Testing XGB - Titanic - V1 - LLaMa-3 - 4bit pipeline [Does not work]

## Model response

Here's a simple example:

```Python
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load the dataset
titanic_data = pd.read_csv('titanic_data.csv')

# Convert categorical columns to numerical columns
titanic_data['Sex'] = pd.get_dummies(titanic_data['Sex'], drop_first=True)
titanic_data['Embarked'] = pd.get_dummies(titanic_data['Embarked'], drop_first=True)

# Split the dataset into training and testing sets
X = titanic_data.drop('Survived', axis=1)
y = titanic_data['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost model
params = {'max_depth': 3, 'eta': 0.1, 'objective': 'binary:logistic', 'eval_metric': 'auc', 'n_estimators': 100}
xgb_model = xgb.XGBClassifier(**params)
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Make predictions on the entire dataset
y_pred = xgb_model.predict(X)

# Save the predictions to a new column in the dataset
titanic_data['Survived_Predicted'] = y_pred

# Save the dataset to a new CSV file
titanic_data.to_csv('titanic_data_with_predictions.csv', index=False)
```

This code assumes that the dataset is stored in a CSV file named 'titanic_data.csv' and that the target column is named 'Survived'. It also assumes that the categorical columns 'Sex' and 'Embarked' are one-hot encoded, and that the 'Survived' column is converted to a binary classification problem (0 for non-survived, 1 for survived). The model is trained on the training set and evaluated on the test set. The predicted probabilities are saved to a new column in the dataset and the dataset is saved to a new CSV file named 'titanic_data_with_predictions.csv'. 

Remember to replace 'titanic_data.csv' with the actual path to your dataset file. Also, you may need to adjust the hyperparameters of the XGBoost model to achieve the best performance on your specific dataset. 

Note: The above code is a simple example and may not produce the best results for your specific task. You may need to preprocess the data further, tune the hyperparameters, or try different machine learning algorithms to achieve the best results. 

Also, the dataset provided is a sample dataset, you should use your own dataset for this task. 

Remember to install the necessary libraries by running `pip install pandas xgboost scikit-learn` before running the code. 


## Pipeline testing

Manual modification: dataset loading, preprocessing transformed to a function
 
!! ValueError: Columns must be same length as key - error in get_dummies

The pipeline does not account for filling missing values 

!! ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, The experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Name: object, Ticket: object, Cabin: object

**Does not work**

In [68]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load the dataset
train_df = pd.read_csv('/home/stas/vaML/titanic/train.csv')
test_df = pd.read_csv('/home/stas/vaML/titanic/test.csv')
res = pd.read_csv('/home/stas/vaML/titanic/gender_submission.csv')

# Convert categorical columns to numerical columns

def preprocess(titanic_data):
    
    titanic_data['Sex'] = pd.get_dummies(titanic_data['Sex'], drop_first=True)

    # titanic_data['Embarked'] = pd.get_dummies(titanic_data['Embarked'], drop_first=True) - error
    
    ### manual fix
    
    dummies = pd.get_dummies(titanic_data['Embarked'], drop_first=True)
    titanic_data = titanic_data.drop('Embarked',axis = 1)
    # Join the encoded df
    titanic_data = titanic_data.join(dummies)
    
    ### manual fix
    
    return titanic_data

train_df = preprocess(train_df)
test_df = preprocess(test_df)
test_df

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Q,S
0,892,3,"Kelly, Mr. James",1,34.5,0,0,330911,7.8292,NaN,1,0
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",0,47.0,1,0,363272,7.0000,NaN,0,1
2,894,2,"Myles, Mr. Thomas Francis",1,62.0,0,0,240276,9.6875,NaN,1,0
3,895,3,"Wirz, Mr. Albert",1,27.0,0,0,315154,8.6625,NaN,0,1
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",0,22.0,1,1,3101298,12.2875,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",1,NaN,0,0,A.5. 3236,8.0500,NaN,0,1
414,1306,1,"Oliva y Ocana, Dona. Fermina",0,39.0,0,0,PC 17758,108.9000,C105,0,0
415,1307,3,"Saether, Mr. Simon Sivertsen",1,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,0,1
416,1308,3,"Ware, Mr. Frederick",1,NaN,0,0,359309,8.0500,NaN,0,1


In [69]:


# Split the dataset into training and testing sets
X_train = train_df.drop(['Survived', 'PassengerId'], axis=1)
y_train = train_df['Survived']

X_test = test_df.drop('PassengerId', axis=1)
y_test = res.drop('PassengerId', axis=1)

# Train an XGBoost model
params = {'max_depth': 3, 'eta': 0.1, 'objective': 'binary:logistic', 'eval_metric': 'auc', 'n_estimators': 100}
xgb_model = xgb.XGBClassifier(**params)
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Make predictions on the entire dataset
y_pred = xgb_model.predict(X_test)

# Print classification report
print(classification_report(y_test, y_pred))

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, The experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Name: object, Ticket: object, Cabin: object

In [67]:
y_test

,Survived
0,0
1,1
2,0
3,0
4,1
...,...
413,0
414,1
415,0
416,0


In [64]:
y_pred

array([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1,
       1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0,
       0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0,
       0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0,
       0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0,

# Testing XGB - Titanic - V2 - GPT-4o pipeline [Works from the box, acc = 0.89]

## Model response

```python
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

# Load datasets
train_data = pd.read_csv('/home/stas/vaML/titanic/train.csv')
test_data = pd.read_csv('/home/stas/vaML/titanic/test.csv')
submission_data = pd.read_csv('/home/stas/vaML/titanic/gender_submission.csv')

# Handle missing values
train_data['Age'].fillna(train_data['Age'].median(), inplace=True)
train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)
test_data['Age'].fillna(test_data['Age'].median(), inplace=True)
test_data['Fare'].fillna(test_data['Fare'].median(), inplace=True)
test_data['Embarked'].fillna(test_data['Embarked'].mode()[0], inplace=True)

# Encode categorical columns
label_enc = LabelEncoder()
train_data['Sex'] = label_enc.fit_transform(train_data['Sex'])
test_data['Sex'] = label_enc.transform(test_data['Sex'])

onehot_enc = OneHotEncoder(drop='first', sparse=False)
embarked_train = onehot_enc.fit_transform(train_data[['Embarked']])
embarked_test = onehot_enc.transform(test_data[['Embarked']])
embarked_train_df = pd.DataFrame(embarked_train, columns=onehot_enc.get_feature_names_out(['Embarked']))
embarked_test_df = pd.DataFrame(embarked_test, columns=onehot_enc.get_feature_names_out(['Embarked']))

train_data = pd.concat([train_data.drop('Embarked', axis=1), embarked_train_df], axis=1)
test_data = pd.concat([test_data.drop('Embarked', axis=1), embarked_test_df], axis=1)

# Define features and target
X_train = train_data.drop(['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin'], axis=1)
y_train = train_data['Survived']
X_test = test_data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
y_test = submission_data['Survived']

# Train the model
model = XGBClassifier(
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    random_state=42
)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print metrics
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
```python

## Pipeline testing

**Accuracy: 0.89**

**Runs perfectly**

XGB params:
```
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    random_state=42
```

In [71]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

# Load datasets
train_data = pd.read_csv('/home/stas/vaML/titanic/train.csv')
test_data = pd.read_csv('/home/stas/vaML/titanic/test.csv')
submission_data = pd.read_csv('/home/stas/vaML/titanic/gender_submission.csv')

# Handle missing values
train_data['Age'].fillna(train_data['Age'].median(), inplace=True)
train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)
test_data['Age'].fillna(test_data['Age'].median(), inplace=True)
test_data['Fare'].fillna(test_data['Fare'].median(), inplace=True)
test_data['Embarked'].fillna(test_data['Embarked'].mode()[0], inplace=True)

# Encode categorical columns
label_enc = LabelEncoder()
train_data['Sex'] = label_enc.fit_transform(train_data['Sex'])
test_data['Sex'] = label_enc.transform(test_data['Sex'])

onehot_enc = OneHotEncoder(drop='first', sparse=False)
embarked_train = onehot_enc.fit_transform(train_data[['Embarked']])
embarked_test = onehot_enc.transform(test_data[['Embarked']])
embarked_train_df = pd.DataFrame(embarked_train, columns=onehot_enc.get_feature_names_out(['Embarked']))
embarked_test_df = pd.DataFrame(embarked_test, columns=onehot_enc.get_feature_names_out(['Embarked']))

train_data = pd.concat([train_data.drop('Embarked', axis=1), embarked_train_df], axis=1)
test_data = pd.concat([test_data.drop('Embarked', axis=1), embarked_test_df], axis=1)

# Define features and target
X_train = train_data.drop(['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin'], axis=1)
y_train = train_data['Survived']
X_test = test_data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
y_test = submission_data['Survived']

# Train the model
model = XGBClassifier(
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    random_state=42
)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print metrics
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')


Accuracy: 0.8875598086124402
Precision: 0.8620689655172413
Recall: 0.8223684210526315
F1 Score: 0.8417508417508416
Precision: 0.8620689655172413
Recall: 0.8223684210526315
F1 Score: 0.8417508417508416


/home/ubuntu/anaconda3/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


# Testing XGB - Titanic - V2 - LLaMa-3 - 4bit pipeline [Light fix, acc = 0.83]

## Model response

```python
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load the dataset
train_data = pd.read_csv('/home/stas/vaML/titanic/train.csv')
test_data = pd.read_csv('/home/stas/vaML/titanic/test.csv')
submission_data = pd.read_csv('/home/stas/vaML/titanic/gender_submission.csv')

# Define the preprocessing steps
numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Sex', 'Embarked']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

# Fit the preprocessing steps
X = preprocessor.fit_transform(train_data)

# Define the XGBoost parameters
params = {
    'objective': 'binary:logistic',
   'max_depth': 3,
    'eval_metric': 'auc',
    'n_estimators': 1000,
    'learning_rate': 0.1,
    'gamma': 0.1,
   'subsample': 0.8,
    'colsample_bytree': 0.8,
   'min_child_samples': 5,
    'random_state': 42
}

# Train the XGBoost model
xgb_model = xgb.XGBClassifier(**params)
xgb_model.fit(X, train_data['Survived'])

# Make predictions on the test data
y_pred = xgb_model.predict(preprocessor.transform(test_data))

# Calculate performance metrics
y_true = submission_data['Survived']
print('Accuracy:', accuracy_score(y_true, y_pred))
print('Classification Report:')
print(classification_report(y_true, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_true, y_pred))
```python
```python
Accuracy: 0.8192
Classification Report:
              Precision    Recall  f1-score   support

           0       0.84      0.92

## Pipeline testing

**Accuracy: 0.83**

!! NameError: name 'StandardScaler' is not defined

Manually fixed standradscaler import

**Weirdly predicted own accuracy**

XGB params:
```
    'objective': 'binary:logistic',
   'max_depth': 3,
    'eval_metric': 'auc',
    'n_estimators': 1000,
    'learning_rate': 0.1,
    'gamma': 0.1,
   'subsample': 0.8,
    'colsample_bytree': 0.8,
   'min_child_samples': 5,
    'random_state': 42
```

In [75]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load the dataset
train_data = pd.read_csv('/home/stas/vaML/titanic/train.csv')
test_data = pd.read_csv('/home/stas/vaML/titanic/test.csv')
submission_data = pd.read_csv('/home/stas/vaML/titanic/gender_submission.csv')

# Define the preprocessing steps
numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Sex', 'Embarked']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

# Fit the preprocessing steps
X = preprocessor.fit_transform(train_data)

# Define the XGBoost parameters
params = {
    'objective': 'binary:logistic',
   'max_depth': 3,
    'eval_metric': 'auc',
    'n_estimators': 1000,
    'learning_rate': 0.1,
    'gamma': 0.1,
   'subsample': 0.8,
    'colsample_bytree': 0.8,
   'min_child_samples': 5,
    'random_state': 42
}

# Train the XGBoost model
xgb_model = xgb.XGBClassifier(**params)
xgb_model.fit(X, train_data['Survived'])

# Make predictions on the test data
y_pred = xgb_model.predict(preprocessor.transform(test_data))

# Calculate performance metrics
y_true = submission_data['Survived']
print('Accuracy:', accuracy_score(y_true, y_pred))
print('Classification Report:')
print(classification_report(y_true, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_true, y_pred))

Accuracy: 0.8349282296650717
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.88      0.87       266
           1       0.78      0.76      0.77       152

    accuracy                           0.83       418
   macro avg       0.82      0.82      0.82       418
weighted avg       0.83      0.83      0.83       418

Confusion Matrix:
[[234  32]
 [ 37 115]]


/home/ubuntu/anaconda3/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [23:05:49] WARNING: /workspace/src/learner.cc:742: 
Parameters: { "min_child_samples" } are not used.

  warnings.warn(smsg, UserWarning)
